# Estimación de un sistema de demanda de energía residencial con regresión aparentemente no relacionada

## Resumen ejecutivo

Una compañía de servicios públicos regional estima un **sistema de demanda de energía** residencial de dos ecuaciones — las participaciones presupuestarias de electricidad y gas natural en función del precio propio, el precio cruzado y el ingreso del hogar — utilizando **PROC SYSLIN** con el método de regresión aparentemente no relacionada (SUR). Las dos ecuaciones de participación comparten un presupuesto común del hogar, por lo que sus errores están correlacionados de forma cruzada; SUR estima las ecuaciones de forma conjunta para explotar esa correlación, recuperando efectos de precio propio y cruzado coherentes en signo y proporcionando la covarianza entre ecuaciones y las pruebas de restricción que un analista de demanda necesita.

## Fuentes de datos

| Conjunto de datos | Filas | Grano | Variables clave | Descripción |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | una fila por observación mensual del mercado | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Panel sintético mensual del mercado residencial de energía. `lp_elec`/`lp_gas` son los precios reales logarítmicos de la electricidad y el gas natural; `lincome` es el logaritmo del ingreso real del hogar; `w_elec`/`w_gas` son las participaciones presupuestarias de gasto en electricidad y gas natural, generadas a partir de una estructura de demanda de tipo AIDS conocida más ruido entre ecuaciones correlacionado. |

# Estimación de un sistema de demanda de energía residencial con regresión aparentemente no relacionada

Una compañía regional de gas y electricidad quiere entender cómo los hogares reasignan su presupuesto energético entre **electricidad** y **gas natural** a medida que cambian los precios relativos y el ingreso. El marco natural es un **sistema de demanda**: un conjunto de ecuaciones de participación presupuestaria estimadas de forma conjunta.

Dos características hacen que la estimación conjunta sea la herramienta adecuada:

- Las ecuaciones de participación se nutren de un presupuesto común del hogar, por lo que sus errores están **correlacionados de forma cruzada** — cuando un hogar gasta más en electricidad, gasta menos en gas. Estimar las ecuaciones juntas con **regresión aparentemente no relacionada (SUR)** aprovecha esa correlación para ganar eficiencia.
- La teoría económica impone **restricciones lineales** (agregación, homogeneidad, simetría) que un estimador de sistema puede imponer directamente.

`PROC SYSLIN` con la opción `SUR` está diseñado exactamente para esta situación. Ajusta cada ecuación de participación, estima la covarianza de los errores entre ecuaciones y reajusta el sistema con esa covarianza para entregar elasticidades eficientes y coherentes con la teoría — junto con la matriz de covarianza entre modelos y las pruebas conjuntas de restricción.

En este cuaderno:

1. Generamos un panel sintético mensual del mercado de energía a partir de una estructura de participación conocida.
2. Ajustamos un sistema de participación de dos ecuaciones con SUR.
3. Comparamos el ajuste conjunto SUR con MCO ecuación por ecuación.
4. Imponemos una restricción de homogeneidad y leemos su prueba F conjunta.
5. Extraemos la tabla de coeficientes para el informe de elasticidades.

## Paso 1 — Generar un panel sintético del mercado de energía

Simulamos 60 observaciones mensuales de un pequeño **sistema de demanda de energía de dos bienes** (electricidad y gas natural). El proceso generador de datos es un modelo de participación presupuestaria linealizado, de tipo AIDS:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Incorporamos deliberadamente **correlación entre ecuaciones**: cuando los hogares gastan más en electricidad gastan menos en gas, por lo que `e1` y `e2` están correlacionados negativamente. Un factor común de precio del mercado energético también impulsa juntos ambos precios logarítmicos. Estas son las características que recompensan la estimación conjunta SUR frente al ajuste de cada ecuación por separado.

In [1]:
DATOS energy;
   LLAMAR streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   HACER month = 1 HASTA 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      SALIDA;
   END;

   MANTENER month lp_elec lp_gas lincome w_elec w_gas;
EJECUTAR;

PROCEDIMIENTO MEDIAS DATOS=energy n mean std MIN MAX maxdec=3;
   VAR w_elec w_gas lp_elec lp_gas lincome;
EJECUTAR;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 2 — Estimación SUR de referencia del sistema de demanda

Estimamos ambas ecuaciones de participación de forma conjunta. La opción `SUR` indica a `PROC SYSLIN` que estime la covarianza de los errores entre ecuaciones y la use para un ajuste eficiente de MCG factible. Dos sentencias `MODEL` etiquetadas (`elec` y `gas`) definen el sistema; cada una regresa una participación presupuestaria sobre los dos precios logarítmicos y el ingreso logarítmico.

SYSLIN informa las estimaciones de parámetros de cada ecuación y, al final, la **matriz de covarianza entre modelos** (Cross-Model Covariance Matrix) — la covarianza estimada de los errores de las dos ecuaciones que SUR explota.

In [2]:
PROCEDIMIENTO syslin DATOS=energy ON;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EJECUTAR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Paso 3 — Comparar con MCO ecuación por ecuación

Para ver qué nos aporta SUR, reajustamos las mismas dos ecuaciones con mínimos cuadrados ordinarios (el método por defecto, una ecuación a la vez). MCO ignora la correlación de errores entre ecuaciones, por lo que es consistente pero menos eficiente que SUR cuando esa correlación es distinta de cero — como lo es aquí por construcción.

Comparar las dos tablas de coeficientes muestra cómo se desplazan las estimaciones y sus errores estándar una vez que se tiene en cuenta la estructura del sistema.

In [3]:
PROCEDIMIENTO syslin DATOS=energy ols;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EJECUTAR;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Paso 4 — Imponer la teoría económica y probarla

La teoría de la demanda dice que los efectos nominales de precio/ingreso deben cumplir la **homogeneidad de grado cero**: escalar todos los precios y el ingreso deja la demanda real sin cambios, por lo que los coeficientes de precio e ingreso dentro de una ecuación suman cero. Imponemos eso sobre la participación de electricidad con una sentencia `RESTRICT`.

SYSLIN reajusta el sistema sujeto a la restricción e informa una prueba F de **resultados de restricción** (Restriction Results) sobre si la restricción es consistente con los datos — una prueba conjunta directa de la hipótesis de homogeneidad.

In [4]:
PROCEDIMIENTO syslin DATOS=energy ON;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
EJECUTAR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Paso 5 — Capturar las estimaciones para el informe de elasticidades

Para un informe final, persistimos los coeficientes convergidos con `OUTEST=`. El conjunto de datos resultante lleva una fila por ecuación con las estimaciones del intercepto y de las pendientes más los estadísticos de ajuste, que alimentan los cálculos de elasticidad de precio propio y cruzado de la compañía en las participaciones medias muestrales. Lo imprimimos para dejar constancia.

In [5]:
PROCEDIMIENTO syslin DATOS=energy ON outest=demand_est;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=demand_est noobs;
   TÍTULO "SUR demand-system coefficient estimates";
EJECUTAR;
TÍTULO;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretación de los resultados

**Coherencia de signos.** Las estimaciones SUR recuperan la estructura de sustitución incorporada en los datos. En la ecuación de participación de electricidad, el **coeficiente del precio propio logarítmico es positivo** (`LP_ELEC` = 0.112) mientras que el **coeficiente del precio cruzado es negativo** (`LP_GAS` = -0.098); en la ecuación de participación de gas el patrón lo refleja (propio `LP_GAS` = 0.114, cruzado `LP_ELEC` = -0.075). Cada efecto de precio propio y cruzado es estadísticamente significativo (cada `Pr > |t|` por debajo de 0.005), por lo que los signos de sustitución están firmemente identificados y no son artefactos de un ajuste ruidoso.

**SUR explota la correlación entre ecuaciones.** La **matriz de covarianza entre modelos** muestra un valor fuera de la diagonal negativo (-0.000068): los hogares intercambian gasto en electricidad por gasto en gas, exactamente como se construyó. Como esa covarianza es distinta de cero, la estimación conjunta SUR es más eficiente que ajustar cada ecuación por separado — los errores estándar de SUR en el Paso 2 son uniformemente algo menores que los errores estándar de MCO ecuación por ecuación en el Paso 3 (por ejemplo, el error estándar de `LP_ELEC` del gas cae de 0.0264 con MCO a 0.0255 con SUR), mientras que las estimaciones puntuales no cambian. Esa inferencia más ajustada es la recompensa de modelar el sistema en lugar de dos regresiones separadas.

**La teoría se sostiene.** Imponer la **homogeneidad de grado cero** sobre la participación de electricidad (coeficientes de precio e ingreso sumando cero) mediante `RESTRICT` produjo una prueba F de resultados de restricción de 2.14 con `Pr > F` = 0.149. La restricción **no se rechaza**, por lo que la predicción de homogeneidad de la teoría de la demanda del consumidor es consistente con esta muestra — la compañía puede llevar las estimaciones restringidas y coherentes con la teoría al informe con confianza.

**Uso operativo.** La tabla `OUTEST=` persiste una fila por ecuación con las estimaciones del intercepto y de las pendientes y los estadísticos de ajuste. Esos coeficientes se convierten directamente en elasticidades de precio propio y cruzado y una elasticidad de ingreso (gasto) en las participaciones medias muestrales (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 del Paso 1), dando a la compañía una base defendible y consistente con la teoría para la previsión de demanda y los escenarios de diseño tarifario.